# Real Pipeline Benchmark (Kaggle GPU)

Runs the repo's **actual production pipeline** — `pipeline.py::run_real_pipeline()` — against
real HaluEval data and a real HuggingFace model, at a scale that isn't practical on a laptop CPU.

Unlike `pytorch_dl_testbed.ipynb` (a synthetic sandbox that deliberately avoids importing repo
code), this notebook clones the repo and calls its real modules directly:
`detector.py`, `feature_engineer.py`, `entropy_baselines.py`, `calibrated_entropy_detector.py`,
`blackbox_detector.py`, `data_generator.py`, `pipeline.py`. Every number this notebook produces
is a real benchmark of the actual detectors, not a proxy.

**What it reports** (via `pipeline.py::run_real_pipeline`'s `results_path` argument):
- Stratified 5-fold CV AUROC + 95% CI for every detector: Logistic Regression, MLP,
  `CalibratedEntropyDetector`, `BlackBoxEntropyDetector`.
- Held-out test metrics (accuracy/precision/recall/F1/FPR) for LogReg, MLP, and BiLSTM.
- Feature-family ablation (which of entropy / lookback / frequency / spectral / cross-layer-KL /
  single-pass-token-entropy actually drives detection).
- Top-10 feature importances (logistic regression weights).

**Requirements:** GPU + internet access (both enabled in `kernel-metadata.json`).


In [ ]:
#@title 1. Clone the repo and add it to sys.path
import os, sys, subprocess

os.chdir("/kaggle/working")

REPO = "Language-Model-Hallucination-Detection-via-Entropy-Divergence"
REPO_URL = f"https://github.com/A-Kuo/{REPO}.git"

if os.path.exists(REPO):
    import shutil
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

REPO_ROOT = os.path.join("/kaggle/working", REPO)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo cloned to:", REPO_ROOT)


In [ ]:
#@title 2. Install the one dependency Kaggle does not preinstall
# torch + transformers already ship on Kaggle's GPU image. `datasets` (for
# HaluEval) usually does too, but install defensively in case it does not.
import importlib
if importlib.util.find_spec("datasets") is None:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  (device count: {torch.cuda.device_count()})")


In [ ]:
#@title 3. Import the real repo modules
from data_generator import DataGenerator
from pipeline import run_real_pipeline

print("Imported DataGenerator, run_real_pipeline from the cloned repo.")


## Configuration

`MODEL_NAME` defaults to a step up from the repo's CPU-friendly default (`pythia-160m`) since
GPU makes a bigger model practical. `NUM_SAMPLES` is far larger than what's reasonable to run
locally on CPU — this is the point of running it here instead.


In [ ]:
#@title 4. Configuration — tune these for your GPU budget
MODEL_NAME = "EleutherAI/pythia-410m"   # bump to pythia-1.4b / phi-2 / etc. if GPU memory allows
NUM_SAMPLES = 3000                       # HaluEval has 10k rows; this draws up to NUM_SAMPLES/2 of each label
SEED = 42
RESULTS_PATH = "/kaggle/working/real_pipeline_benchmark.json"

print(f"model={MODEL_NAME}  num_samples={NUM_SAMPLES}  seed={SEED}")


In [ ]:
#@title 5. Load real HaluEval data (no API needed)
samples = DataGenerator.from_halueval(num_samples=NUM_SAMPLES, seed=SEED)
print(f"Loaded {len(samples)} labeled samples "
      f"({sum(1 for s in samples if s.label == 'hallucinated')} hallucinated, "
      f"{sum(1 for s in samples if s.label == 'correct')} correct)")


In [ ]:
#@title 6. Run the real pipeline: feature extraction + CV + ablation for every detector
summary = run_real_pipeline(
    samples,
    model_name=MODEL_NAME,
    seed=SEED,
    results_path=RESULTS_PATH,
)


## Reading the results

`summary` (and the JSON file at `RESULTS_PATH`, which lands in `notebooks/results/` once the
Kaggle GPU workflow pulls it back into the repo) contains:

- `cv_results` — AUROC + 95% CI per detector (`logistic`, `mlp`, `calibrated_entropy`, `blackbox_topk`).
- `held_out` — full metrics (accuracy/precision/recall/F1/FPR) for the held-out split.
- `bilstm` — same, for the per-layer-sequence BiLSTM classifier (`null` if PyTorch was unavailable).
- `ablation` — full-model AUROC plus the AUROC drop from removing each feature family.
- `feature_importance_top10` — logistic regression's top 10 weighted features.

### Using these numbers to improve the project

- If `calibrated_entropy` or `blackbox_topk` now beats `logistic` at this scale, that is a real
  signal worth acting on (the synthetic sandbox notebook can only suggest directions; this
  notebook's numbers are the real thing).
- Compare `ablation["families"]` deltas here against the numbers already committed in
  `paper/paper.tex` / `COLABS.md` — those were generated on `pythia-160m` at `num_samples=500`;
  a large gap at this bigger scale/model is worth writing up.
- If `bilstm` still underperforms `logistic` at this larger, cleaner scale (the label-alignment
  and context-length bugs that used to corrupt this comparison are now fixed), that is stronger
  evidence the sequence model genuinely is not earning its complexity here — not just noise.


## What "AUROC + 95% CI" actually means here

**Why cross-validation instead of one train/test split?** With HaluEval-scale data, a single
70/30 split's AUROC can swing noticeably just from which examples happened to land in the test
set. `run_real_pipeline` reports both: a single held-out split (`held_out`, for a concrete
confusion matrix) *and* `cv_results`, a stratified 5-fold estimate — five different train/test
partitions of the same data, each contributing out-of-fold predictions, pooled into one AUROC.
This is a better estimate of "how this detector performs on unseen data in general" than any
single split, because it isn't sensitive to one lucky or unlucky split.

**The Math.** For each of the 5 folds $f$, the detector is trained on the other 4 folds and
scores the held-out fold $f$; every example ends up with exactly one out-of-fold prediction
$\hat{p}_i$. AUROC on the pooled predictions is computed via the Mann-Whitney U identity —
the same quantity, and the same formula, as README.md's "A note on AUROC" section:

$$
\mathrm{AUROC} = \frac{1}{n_{+} n_{-}} \sum_{i \in \text{pos}} \sum_{j \in \text{neg}} \mathbb{1}[\hat{p}_i > \hat{p}_j]
$$

The 95% CI comes from **bootstrap resampling**: redraw $N$ examples with replacement from the
pooled $(\hat{p}_i, y_i)$ pairs 1000 times, recompute AUROC on each resample, and report the
2.5th/97.5th percentiles of that distribution (`pipeline.py::bootstrap_auroc_ci`). A narrow CI
means the AUROC estimate is stable across resamples of this dataset; a wide CI (common at small
`NUM_SAMPLES`) means don't read too much into the point estimate alone — this is exactly why
running this notebook at a bigger `NUM_SAMPLES` than is practical on a laptop CPU is the point.

**The Code.** `pipeline.py::stratified_kfold_cv` (the fold loop + Mann-Whitney AUROC) and
`bootstrap_auroc_ci` (the CI) — both called once per detector inside `run_real_pipeline`.